# 임베딩(밀집벡터)와 트랜스포머에 대해 생각해 보자

기존의 Word2Vec같은 모델들은 단어를 **'정적(Static)'**으로 임베딩한다. 

즉, 문장에서 어떤 역할을 하든 그 단어는 항상 똑같은 숫자로 변환된다. 

하지만 트랜스포머의 인코더는 **셀프 어텐션(Self-Attention)**을 통해 같은 단어라도 주변에 어떤 단어들이 있느냐에 따라 벡터값을 실시간으로 계산한다. 
> 이를 '문맥화된 표현(Contextualized Representation)'이라고 부른다
>

**BERT의 목적 = 최고의 문맥 임베딩 만들기**

즉, BERT는 "이 단어가 이 문장 안에서 정확히 어떤 의미로 쓰였는지 가장 잘 설명하는 벡터를 만들어내자!"라는 목적에 올인한 모델이다.

그런데, 인콛에 들어가기 직전, 토큰들은 아직 문맥을 모르는 고정된 벡터 상태이다. 여기에 포지셔널 인코딩이 더해지고 인코더 layer를 통과한다

layer를 통과하면 Contextualized Embedding 즉, 인코더 내부의 셀프 어텐션 층을 거치면서 각 벡터가 문맥을 반영하게 된다. 

## 근데 word2vec도 문맥을 반영하는거 아닌가?

왜 word2vec은 문맥을 반영하도록 학습했는데, 트랜스포머와 달리 정적이라고 할까?

> 정적 이라는 말은 모델이 학습할 떄 문맥을 보지 않는다는 뜻이 아니라, 학습이 끝난 후 '사용'할 때 문맥에 따라 값이 변하지 않는다는 것이다
>
1. "다른 코퍼스에서 학습하면 벡터가 다르지 않나?" (맞습니다!)
금융 뉴스 코퍼스로 학습할 떄 '경제'라는 단어와 정치 기사 코퍼스로 학습한 '경제'라는 단어의 벡터는 분명히 다르다. 하지만, 금융 뉴스 모델로 학습한 word2vec 모델 또는 정치 뉴스로 학습한 모델이 일단 완성되고 나면, 그 모델 안에서 '경제'라는 단어는 언제나 어떤 문장에서나 똑같은 고정된 벡터값을 가지게 된다. 


그러므로, 한 번 학습된 (문맥을 반영해여) word2vec 모델은 입력 문장의 상황에 따라 단어의 벡터가 실시간으로 변하지 않는다.   
-> **그래서 룩업 방식이다**


하지만, BERT나 Transformer는 셀프 어텐션 과정을 거치면서 같은 단어라도 따로따로 계산할 수 있다. (하나의 모델 안에서도 다르게 계산한다!)

Word2Vec이 학습 시 문맥을 본다는 것은 **"단어 간의 일반적인 관계"**를 파악한다는 의미에 가깝다.

'사과' 근처에는 '맛있다', '빨갛다'가 자주 나온다는 전역적인 통계를 학습하는 것이지,

지금 내 눈앞에 주어진 특정한 문장 안에서의 개별적인 역할을 파악하는 것은 아니다.

> 그러므로 word2vec은 내가 분석하려는 데이터의 도메인과 모델이 학습한 데이터의 성격이 일치할 수록 성능이 훨씬 잘 나올 것이다. (정치 기사로 학습한 모델과 경제 기사로 학습한 모델을 생각해 보자)

여기서 Word2Vec과 BERT(트랜스포머)를 가르는 가장 결정적인 차이가 있다.

word2vec은 단어 사전을 보고 답을 찾아오는 룩업 테이블 방식이다. 하지만, BERT는 문장을 읽으며 매번 값을 만들어내는 **계산**과정이다.

즉, 하나의 모델 안에서도 정치 맥락의 경제와 경제 맥락의 경제가 다르게 벡터화 된다. 


이렇게 되면 모델을 갈아 끼울 필요가 없이 BERT나 Transformer 모델 하나 안에서 해결이 가능하다.

기존(Word2Vec): 정치 데이터로 학습한 모델 A, 경제 데이터로 학습한 모델 B를 따로 관리해야 하거나, 모델 하나가 '경제'라는 단어의 여러 의미를 하나로 뭉뚱그려 학습해야 했습니다.

현재(BERT): 위키피디아나 뉴스 등 방대한 데이터를 한꺼번에 학습한 모델 하나만 있으면, 그 모델이 알아서 문맥을 파악해 상황에 맞는 벡터를 생성합니다. 이를 범용 언어 모델이라고 부르는 이유이기도 합니다.

### 임베딩 행렬

우리는 이전에 word2vec이 임베딩 행렬을 만든다고 배웠다.

word2vec과 같은 정적 임베딩은 임베딩 행렬(단어개수 V * 벡터차원 D)을 가지고 있다. 
그래서 '경제'라는 단어가 들어오면 행려렝 n번째 행에 저장된 벡터를 그냥 꺼내서 준다.

그러므로, word2vec 모델 학습 시 이 임베딩 행렬을 최적화 하고 사용할 떄에는 추가 연산 전혀 없이 사용한다 즉, 언제나 같다


하지만, BERT나 Transformer는?

> 재료 준비 (Input Embedding Matrix): 일단 '경제'라는 단어가 들어오면, BERT도   자기 내부의 임베딩 행렬에서 고정된 벡터값을 꺼내옵니다. (이 시점에서는 Word2Vec과 똑같습니다.)

> 조리 시작 (Self-Attention Layers): 꺼내온 벡터가 인코더 층을 통과하기 시작합니다. 여기서 주변 단어(정치적 단어들 혹은 금융 단어들)의 벡터와 서로 섞이고 연산되는 과정을 거칩니다.

> 최종 요리 완성 (Hidden States): 여러 층을 다 통과하고 나온 최종 출력 벡터가 바로 우리가 말하는 '문맥 반영 임베딩'입니다.


BERT: 기본 재료(행렬)는 같아도, 어떤 양념(주변 단어)과 함께 볶느냐(어텐션)에 따라 매번 다른 맛(최종 벡터)을 낼 수 있음.


*Transforemr나 BERT 같은 모델들도 처음에는 자신만의 임베딩 행렬이 있다. BERT-base 모델 기준 약 30,000개의 토큰을 가지고 있다. 이걸 768 차원으로 표현하는 임베딩 행렬이 있다. 여기까지가 Token Embeddings인데, 이것만 한다면 Word2vec이랑 동일한다. 근데 BERT의 경우를 생각해 보면 Segment Embeddings, Position Embeddings을 입히고 인코딩 layer에 통과시켜서 문맥을 반영한 벡터로 만든다. 즉, Transformer나 BERT가 Word2vec 보다 훨씬 문맥 반영 잘한다.*